In [ ]:
import modified_didppy as m_dp
import numpy as np
from scipy.sparse.csgraph import minimum_spanning_tree
import itertools
import sys
import os

# 1. Simple problem

In [ ]:
print(f"--- Testing DIDPPy ---")
print(f"Python executable: {sys.executable}")
print(f"DIDPPy module location: {os.path.abspath(m_dp.__file__)}")

# 1. Create a minimal model
model = m_dp.Model()
#
# --- THIS IS THE FIX ---
#
# Create an integer variable 'x'. Set the START STATE to 5.
x = model.add_int_var(target=5)
# Set the GOAL CONDITION to x == 0.
model.add_base_case([x == 0])
#
# --- END OF FIX ---
#
model.add_transition(
    m_dp.Transition(
        name="decrement",
        cost=1 + m_dp.IntExpr.state_cost(), # Cost is 1 per step
        effects=[(x, x - 1)]
    )
)
# This line is not needed for forward search, so we remove it.
# model.target_state[x] = 5 

print("Model created. Start state x=5, Target state x=0.")

# 2. Add a standard Rust expression bound
model.add_dual_bound(x) 
print("Added Rust-based dual bound (returns x)")

# 3. Define your Python function dual bound
def bound_A(state):
    return float(1)
def bound_B(state):
    return float(2)
def my_python_bound(state):
    try:
        val = state[x]
        bound = float(val * 2) # Return a float, as the evaluator expects
        # print(f"Python func called: state[x]={val}, returning bound={bound}")
        bound = bound + bound_A(state) + bound_B(state)
        return bound
    except Exception as e:
        print(f"Error in Python function: {e}")
        return None

print("Defined Python-based dual bound (returns float(x * 2))")

# 4. Instantiate your NEW solver
try:
    # This is your new class from customized_cabs_ver1.rs
    solver = m_dp.CustomDualBoundCABSv1(
        model, 
        dual_bound_func=my_python_bound, 
        quiet=False # Set to False to see the solver logs
    )
    print("\nSuccessfully created CustomDualBoundCABSv1 solver.")
except AttributeError:
    print("\n--- !! ERROR !! ---")
    print("Could not find 'dp.CustomDualBoundCABSv1'.")
    print("This means your Rust code is not installed correctly.")
    print("Please run `maturin develop` in the `didp-rs-dev/didppy` folder.")
    sys.exit(1)
except Exception as e:
    print(f"An error occurred creating the solver: {e}")
    sys.exit(1)

# 5. Run the search
print("Starting search...")
solution = solver.search()

# 6. Verify the result
print("\n--- Search Finished ---")
print(f"Transitions: {[t.name for t in solution.transitions]}")
print(f"Cost: {solution.cost}")

# The cost from x=5 to x=0 is 5 steps of cost 1.
assert solution.cost == 5
print("\n✅ Test Passed: The solution cost is correct!")

SyntaxError: invalid syntax (1401131606.py, line 80)

# 2. Testing with MST bounds of CVRP model

In [7]:
print(f"--- Testing DIDPPy ---")
print(f"Python executable: {sys.executable}")
print(f"DIDPPy module location: {os.path.abspath(m_dp.__file__)}")

# =========================================================
# 1️⃣ Define Data (MOVED TO TOP)
# =========================================================
# Number of locations (0 is depot)
n = 4
# Number of vehicles
m = 2
# Capacity of a vehicle
q = 5
# Weights (d[0] is 0 for depot)
d = [0, 2, 3, 3]

# Distance matrix as a list of lists for the table
distance_list = [
    [0, 3, 4, 5],
    [3, 0, 5, 4],
    [4, 5, 0, 3],
    [5, 4, 3, 0]
]
# Distance matrix as a NumPy array for the Python function
distance_matrix_np = np.array(distance_list)

print("Data defined.")

# =========================================================
# 2️⃣ Define DIDP model
# =========================================================
model = m_dp.Model()

# --- Variables ---
customer = model.add_object_type(number=n)
unvisited_var = model.add_set_var(object_type=customer, target=list(range(1, n)), name = 'unvisited_customers')
location_var = model.add_element_var(object_type=customer, target=0)
load_var = model.add_int_resource_var(target=0, less_is_better=True)
vehicles_var = model.add_int_resource_var(target=1, less_is_better=True)

# --- Tables ---
weight = model.add_int_table(d)
distance_table = model.add_int_table(distance_list)

# --- Base case ---
# Goal is to be unvisited_var is empty AND at the depot (location 0)
model.add_base_case([unvisited_var.is_empty(), location_var == 0])

# --- Transitions ---
for j in range(1, n):
    visit = m_dp.Transition(
        name=f"visit {j}",
        cost=distance_table[location_var, j] + m_dp.IntExpr.state_cost(),
        effects=[
            (unvisited_var, unvisited_var.remove(j)),
            (location_var, j),
            (load_var, load_var + weight[j]),
        ],
        preconditions=[unvisited_var.contains(j), load_var + weight[j] <= q],
    )
    model.add_transition(visit)

for j in range(1, n):
    visit_via_depot = m_dp.Transition(
        name=f"visit {j} with new vehicle",
        cost=distance_table[location_var, 0] + distance_table[0, j] + m_dp.IntExpr.state_cost(),
        effects=[
            (unvisited_var, unvisited_var.remove(j)),
            (location_var, j),
            (load_var, weight[j]), # Load resets to just this customer
            (vehicles_var, vehicles_var + 1),
        ],
        preconditions=[unvisited_var.contains(j), vehicles_var < m],
    )
    model.add_transition(visit_via_depot)

return_to_depot = m_dp.Transition(
    name="return",
    cost=distance_table[location_var, 0] + m_dp.IntExpr.state_cost(),
    effects=[(location_var, 0)],
    preconditions=[unvisited_var.is_empty(), location_var != 0],
)
model.add_transition(return_to_depot)

# --- State constraint ---
# Fixed: Need to use .sum() for a set variable index
model.add_state_constr((m - vehicles_var + 1) * q - load_var >= weight[unvisited_var])

print("DIDP Model created.")

# =========================================================
# 3️⃣ Define Python Dual Bound Function
# =========================================================
def compute_dynamic_mst(state):
    """Compute MST cost for the current state's unvisited customers + depot."""
    try:
        # Get the list of unvisited customer IDs *from the state*
        unvisited_customers_list = list(state[unvisited_var]) 
        
        # Create the list of nodes for the MST (depot 0 + unvisited)
        nodes_for_mst = [0] + unvisited_customers_list
        
        if len(nodes_for_mst) <= 1:
            return 0.0 # No MST needed
        
        # Create the sub-matrix for these nodes
        sub_matrix = distance_matrix_np[np.ix_(nodes_for_mst, nodes_for_mst)]
        
        # Compute and return the MST cost
        mst = minimum_spanning_tree(sub_matrix)
        
        # Return as a float, as our evaluator expects
        return float(mst.sum())
    
    except Exception as e:
        print(f"Error in Python dual bound: {e}")
        return None # Return None on failure

print("Python dual bound function `compute_dynamic_mst` defined.")

# =========================================================
# 4️⃣ Solve model (Using your NEW Custom CABS)
# =========================================================
print("\nInstantiating CustomDualBoundCABSv1...")
try:
    # We do NOT add the bound to the model.
    # We pass the Python function *directly to the solver*
    solver = m_dp.CustomDualBoundCABSv1(
        model, 
        dual_bound_func=compute_dynamic_mst, 
        quiet=False
    )
    print("Successfully created solver. Starting search...")
    solution = solver.search()
    
except AttributeError:
    print("\n--- !! ERROR !! ---")
    print("Could not find 'dp.CustomDualBoundCABSv1'.")
    print("This means your Rust code is not installed correctly.")
    print("Please run `maturin develop` in the `didp-rs-dev/didppy` folder")
    print("and RESTART your Jupyter kernel.")
    sys.exit(1)
except Exception as e:
    print(f"An error occurred creating/running the solver: {e}")
    sys.exit(1)

# =========================================================
# 5️⃣ Print solution and transitions
# =========================================================
print("\n=== Optimal Solution ===")
if solution.cost is not None:
    print(f"Best cost: {solution.cost}")
    print("\n=== Transition sequence ===")
    for t in solution.transitions:
        print(f" - {t.name}")
else:
    print("No solution found.")

print("\n✅ Test finished.")

--- Testing DIDPPy ---
Python executable: c:\Users\ACER\Desktop\Code\0.Thesis implementation\DIDP_custom_search_guidance_local\Thesis_modified_DIDP\venv\Scripts\python.exe
DIDPPy module location: c:\Users\ACER\Desktop\Code\0.Thesis implementation\DIDP_custom_search_guidance_local\Thesis_modified_DIDP\venv\Lib\site-packages\modified_didppy\__init__.py
Data defined.
DIDP Model created.
Python dual bound function `compute_dynamic_mst` defined.

Instantiating CustomDualBoundCABSv1...
Successfully created solver. Starting search...

=== Optimal Solution ===
Best cost: 20

=== Transition sequence ===
 - visit 1
 - visit 3
 - visit 2 with new vehicle
 - return

✅ Test finished.
